# Episode 11: The Consulting Adapter

Every channel is governed by an **adapter** — it decides who may speak, in what order, and when the channel closes. The simplest is `consulting`.

**In this episode you'll learn:** the `consulting` adapter — strict one-question / one-answer, with an automatic close.

## What `consulting` guarantees

`consulting` is a **strict 1Q1R** channel between exactly two participants:

1. The initiator sends exactly one substantive message.
2. The respondent sends exactly one reply.
3. The adapter **auto-closes** the channel with reason `consulting_complete`.

That's the whole contract. No back-and-forth, no ambiguity about when it ends. The adapter *rejects* an out-of-order send or any send after both sides have spoken.

Use it for expert lookups, classification calls, or any "ask one thing, get one answer" exchange where you want a hard guarantee on shape.

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.knowledge import MemoryKnowledgeStore
from autogen.beta.network import (
    EV_CHANNEL_CLOSED,
    EV_TEXT,
    Hub,
    HubClient,
    LocalLink,
    Passport,
    Resume,
)

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

## A consulting call

An `asker` consults an `expert`. Notice we never call `channel.close()` — the adapter closes the channel itself the moment the expert has answered.

In [ ]:
hub = await Hub.open(MemoryKnowledgeStore(), ttl_sweep_interval=0)
link = LocalLink(hub)
asker_hc = HubClient(link, hub=hub)
expert_hc = HubClient(link, hub=hub)

asker = await asker_hc.register(
    Agent("asker", prompt="Ask one focused question and stop.", config=config),
    Passport(name="asker"),
    Resume(),
    attach_plugin=False,
)
expert = await expert_hc.register(
    Agent(
        "expert",
        prompt="You are a database expert. Answer in one clear sentence.",
        config=config,
    ),
    Passport(name="expert"),
    Resume(),
    attach_plugin=False,
)

# consulting = strict one-question / one-reply; the adapter auto-closes
# the channel as soon as the expert has answered.
channel = await asker.open(type="consulting", target="expert")
await channel.send(
    "What problem does database indexing solve?", audience=[expert.agent_id]
)

close_env = await asker.wait_for_channel_event(
    channel_id=channel.channel_id,
    predicate=lambda e: e.event_type == EV_CHANNEL_CLOSED,
    timeout=60.0,
)
print(f"auto-closed with reason: {close_env.event_data.get('reason')!r}")

names = {asker.agent_id: "asker", expert.agent_id: "expert"}
for env in await hub.read_wal(channel.channel_id):
    if env.event_type == EV_TEXT:
        print(f"{names[env.sender_id]}: {env.event_data['text']}")

await asker_hc.close()
await expert_hc.close()
await hub.close()

## Why the strictness helps

The `consulting_complete` close reason is a **guarantee**: if you see it, you know exactly one question was asked and exactly one answer was given. Your orchestration code doesn't have to count messages or guess when to stop.

The adapter also enforces this — try to send a second question and it raises a `ProtocolError`. The shape of the exchange is part of the contract, not a convention you hope everyone follows.

## Additional content

**Expectations.** `consulting` ships strict timeouts: if the respondent doesn't acknowledge the invite within 30s, or doesn't reply within 10 minutes, the Hub auto-closes with an `expectation_violated:...` reason. These are tunable.

**`audience=`.** `channel.send(text, audience=[expert.agent_id])` controls who *sees* the envelope. With two participants it's the obvious choice; in larger channels it's how you scope a message.

**When not to use it.** Need a real back-and-forth? Use `conversation` (Episode 12). Need 3+ participants? Use `discussion` (Episode 13) or `workflow` (Episode 14).

## What's next

`consulting` allows exactly one reply. Episode 12's `conversation` adapter removes that limit — a free-form, two-party dialogue.